In [1]:
#QB ML MODEL - IMPROVED
import pandas as pd
import numpy as np
import warnings
import nflreadpy
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import joblib

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

scaler = MinMaxScaler()
PEAK_AGE_QB = 30

dfFantasy = pd.read_pickle("PickleFiles/final_qb_data.pkl")
dfFantasy.replace([np.inf, -np.inf], np.nan, inplace=True)
for column in dfFantasy.select_dtypes(include=[np.number]).columns:
    dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)

# ---- Add advanced EPA/efficiency features from nflreadpy ----
print("Fetching QB advanced features (passing_epa, pacr) for 2015-2025...")
seasonal_all = nflreadpy.load_player_stats(list(range(2015, 2026)), summary_level='reg').to_pandas()
adv = seasonal_all[['player_id', 'season', 'passing_epa', 'pacr']].copy()
adv['season'] = adv['season'].astype(int)
dfFantasy['season'] = dfFantasy['season'].astype(int)
dfFantasy = dfFantasy.merge(adv, on=['player_id', 'season'], how='left')
print(f"  passing_epa coverage: {dfFantasy['passing_epa'].notna().mean()*100:.0f}%")

def correctData(df, pprTF):
    # Counting stats to convert to per-game
    count_cols = ['completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions',
                  'sacks', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch',
                  'passing_first_downs', 'passing_2pt_conversions',
                  'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                  'rushing_first_downs', 'rushing_2pt_conversions', 'fantasy_points', 'age']
    df.loc[:, 'PPG'] = df['fantasy_points'] / df['GP']
    # ppg_prev = last year's PPG — computed BEFORE shifting (will become prev-year feature after shift)
    df = df.sort_values(['player_display_name', 'season']).reset_index(drop=True)
    df['ppg_prev'] = df['PPG']
    for col in count_cols:
        df.loc[:, col] = df[col] / df['GP']
    # passing_epa is a season total -> divide by GP; pacr is a ratio -> leave as-is
    df.loc[:, 'passing_epa'] = df['passing_epa'] / df['GP'].replace(0, np.nan)
    # pacr: already normalised, do NOT divide
    df = df[df.GP > 7]
    df = df[df.fantasy_points >= 0]
    df = df[df.PPG > 5]
    df = df.sort_values(['player_display_name', 'season']).reset_index(drop=True)
    df['ppg_last_year'] = df.groupby('player_display_name')['PPG'].shift(1).fillna(df['PPG'])
    df['delta_ppg'] = df['PPG'] - df['ppg_last_year']
    return df

def removeUnwanted(dfPos, pos):
    drop_cols = ['season', 'GP', 'season_type', 'fantasy_points',
                 'player_display_name', 'player_id', 'team', 'position']
    return dfPos.drop(columns=drop_cols, errors='ignore')

def makeCorrectShift(df):
    # All stats + advanced features + ppg_prev shifted 1 year back so they become "last year's" features
    shifters = ['season', 'GP', 'season_type', 'age', 'fantasy_points',
                'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions',
                'sacks', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch',
                'passing_first_downs', 'passing_2pt_conversions',
                'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                'rushing_first_downs', 'rushing_2pt_conversions',
                'passing_epa', 'pacr',
                'ppg_prev', 'ppg_last_year', 'delta_ppg']
    # Only shift columns that actually exist in df
    shifters = [c for c in shifters if c in df.columns]
    df[shifters] = df.groupby('player_display_name')[shifters].shift(1)
    return df.dropna()

XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 3, 'min_child_weight': 5,
    'reg_alpha': 0.3, 'reg_lambda': 2.0, 'subsample': 0.75,
    'colsample_bytree': 0.7, 'learning_rate': 0.05,
    'random_state': 42, 'verbosity': 0
}

TRAIN_START_YEARS = [2019, 2018, 2017, 2016, 2015]

def getScaleBack(df):
    return [df['PPG'].min(), df['PPG'].max()]

def machineLearning(df, arr):
    predictors = [c for c in df.columns
                  if c != 'PPG' and 'Unnamed' not in c and c != 'YearsBack']
    x_df = df[predictors]
    y = df['PPG'].values
    split = int(len(y) * 0.8)
    cv = xgb.XGBRegressor(**XGB_PARAMS)
    cv.fit(x_df.iloc[:split], y[:split])
    p = cv.predict(x_df.iloc[split:])
    p_ppg = p * (arr[1] - arr[0]) + arr[0]
    a_ppg = y[split:] * (arr[1] - arr[0]) + arr[0]
    mae = mean_absolute_error(a_ppg, p_ppg)
    corr = 0.0
    if len(a_ppg) > 2:
        corr, _ = pearsonr(p_ppg, a_ppg)
        print(f"  Holdout MAE: {mae:.2f} PPG | r={corr:.3f} | n={len(a_ppg)}")
    final = xgb.XGBRegressor(**XGB_PARAMS)
    final.fit(x_df, y)
    return (mae, corr, final)

print("Training QB models (3 PPR formats) with trial-and-error year ranges...")
for ppr in [0, 1, 2]:
    lbl = {0: 'Standard', 1: 'HalfPPR', 2: 'FullPPR'}[ppr]
    print(f"\nQB {lbl}:")

    dfFC_base = dfFantasy.copy()
    dfFC_base = correctData(dfFC_base, ppr)
    dfFC_base = makeCorrectShift(dfFC_base)

    best_corr = -999
    best_model = None
    best_start = None

    for start_year in TRAIN_START_YEARS:
        dfFC = dfFC_base[dfFC_base['season'] >= start_year].copy()
        dfFC = dfFC.loc[dfFC['season'] != 2012]
        dfFC['age_from_peak'] = dfFC['age'] - PEAK_AGE_QB
        dfFC['age_squared'] = dfFC['age'] ** 2
        dfFC['games_missed'] = (17 - dfFC['GP']).clip(lower=0)
        dfFC = removeUnwanted(dfFC, 'QB')
        dfFC = dfFC.reset_index(drop=True)
        if len(dfFC) < 20:
            print(f"  start_year={start_year}: too few rows ({len(dfFC)}), skipping")
            continue
        scaleQB = getScaleBack(dfFC)
        dfFC_scaled = dfFC.copy()
        dfFC_scaled[dfFC_scaled.columns] = scaler.fit_transform(dfFC_scaled[dfFC_scaled.columns])
        print(f"  start_year={start_year} ({len(dfFC)} rows, {len(dfFC.columns)-1} features):", end=" ")
        mae, corr, model = machineLearning(dfFC_scaled, scaleQB)
        if corr > best_corr:
            best_corr = corr
            best_model = model
            best_start = start_year

    print(f"  Best start_year={best_start} with r={best_corr:.3f}")
    path = {0: "qb models/qbModelNonPPR.joblib",
            1: "qb models/qbModelHalfPPR.joblib",
            2: "qb models/qbModelPPR.joblib"}[ppr]
    joblib.dump(best_model, path)
    print(f"  Saved {path}")


Fetching QB advanced features (passing_epa, pacr) for 2015-2025...
  passing_epa coverage: 97%
Training QB models (3 PPR formats) with trial-and-error year ranges...

QB Standard:
  start_year=2019 (167 rows, 31 features):   Holdout MAE: 2.25 PPG | r=0.669 | n=34
  start_year=2018 (197 rows, 31 features):   Holdout MAE: 2.21 PPG | r=0.691 | n=40
  start_year=2017 (221 rows, 31 features):   Holdout MAE: 2.04 PPG | r=0.711 | n=45
  start_year=2016 (247 rows, 31 features):   Holdout MAE: 2.01 PPG | r=0.706 | n=50
  start_year=2015 (247 rows, 31 features):   Holdout MAE: 2.01 PPG | r=0.706 | n=50
  Best start_year=2017 with r=0.711
  Saved qb models/qbModelNonPPR.joblib

QB HalfPPR:
  start_year=2019 (167 rows, 31 features):   Holdout MAE: 2.25 PPG | r=0.669 | n=34
  start_year=2018 (197 rows, 31 features):   Holdout MAE: 2.21 PPG | r=0.691 | n=40
  start_year=2017 (221 rows, 31 features):   Holdout MAE: 2.04 PPG | r=0.711 | n=45
  start_year=2016 (247 rows, 31 features):   Holdout MAE: 2.

In [2]:
#RB ML MODEL - IMPROVED
import pandas as pd
import numpy as np
import warnings
import nflreadpy
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import joblib

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

scaler = MinMaxScaler()
PEAK_AGE_RB = 25

dfFantasy = pd.read_pickle("PickleFiles/final_rb_data.pkl")
dfFantasy.replace([np.inf, -np.inf], np.nan, inplace=True)
for column in dfFantasy.select_dtypes(include=[np.number]).columns:
    dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)

# ---- Add opportunity + EPA features from nflreadpy ----
print("Fetching RB opportunity/EPA features for 2015-2025...")
seasonal_all = nflreadpy.load_player_stats(list(range(2015, 2026)), summary_level='reg').to_pandas()
opp = seasonal_all[['player_id', 'season', 'target_share',
                     'rushing_epa', 'receiving_epa']].copy()
opp['season'] = opp['season'].astype(int)
dfFantasy['season'] = dfFantasy['season'].astype(int)
dfFantasy = dfFantasy.merge(opp, on=['player_id', 'season'], how='left')
print(f"  target_share coverage: {dfFantasy['target_share'].notna().mean()*100:.0f}%")
print(f"  rushing_epa coverage:  {dfFantasy['rushing_epa'].notna().mean()*100:.0f}%")

def correctData(df, pprTF):
    count_cols = ['carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                  'rushing_first_downs', 'rushing_2pt_conversions', 'receptions', 'targets',
                  'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost',
                  'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs',
                  'receiving_2pt_conversions', 'special_teams_tds', 'fantasy_points', 'rrtd', 'age']
    if pprTF == 0:
        df.loc[:, "fantasy_points"] = df["fantasy_points"] - df["receptions"]
    elif pprTF == 1:
        df.loc[:, "fantasy_points"] = df["fantasy_points"] - (df["receptions"] / 2)
    df.loc[:, 'PPG'] = df['fantasy_points'] / df['GP']
    # ppg_prev before shift
    df = df.sort_values(['player_display_name', 'season']).reset_index(drop=True)
    df['ppg_prev'] = df['PPG']
    for col in count_cols:
        df.loc[:, col] = df[col] / df['GP']
    # EPA totals -> per game; shares -> leave as-is
    gp_safe = df['GP'].replace(0, np.nan)
    df.loc[:, 'rushing_epa'] = df['rushing_epa'] / gp_safe
    df.loc[:, 'receiving_epa'] = df['receiving_epa'] / gp_safe
    # target_share is a ratio — do NOT divide
    df = df[df.GP > 7]
    df = df[df.fantasy_points >= 0]
    df = df[df.PPG > 2]
    df = df.sort_values(['player_display_name', 'season']).reset_index(drop=True)
    df['ppg_last_year'] = df.groupby('player_display_name')['PPG'].shift(1).fillna(df['PPG'])
    df['delta_ppg'] = df['PPG'] - df['ppg_last_year']
    return df

def removeUnwanted(dfPos, pos):
    drop_cols = ['season', 'GP', 'season_type', 'fantasy_points',
                 'player_display_name', 'player_id', 'team', 'position']
    return dfPos.drop(columns=drop_cols, errors='ignore')

def makeCorrectShift(df):
    shifters = ['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
                'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
                'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_2pt_conversions',
                'receptions', 'targets', 'receiving_yards', 'receiving_tds',
                'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch',
                'receiving_first_downs', 'receiving_2pt_conversions', 'special_teams_tds',
                'fantasy_points', 'rrtd',
                'target_share', 'rushing_epa', 'receiving_epa',
                'ppg_prev', 'ppg_last_year', 'delta_ppg']
    shifters = [c for c in shifters if c in df.columns]
    df[shifters] = df.groupby('player_display_name')[shifters].shift(1)
    return df.dropna()

XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 3, 'min_child_weight': 5,
    'reg_alpha': 0.3, 'reg_lambda': 2.0, 'subsample': 0.75,
    'colsample_bytree': 0.7, 'learning_rate': 0.05,
    'random_state': 42, 'verbosity': 0
}

TRAIN_START_YEARS = [2019, 2018, 2017, 2016, 2015]

def getScaleBack(df):
    return [df['PPG'].min(), df['PPG'].max()]

def machineLearning(df, arr):
    predictors = [c for c in df.columns
                  if c != 'PPG' and 'Unnamed' not in c and c != 'YearsBack']
    x_df = df[predictors]
    y = df['PPG'].values
    split = int(len(y) * 0.8)
    cv = xgb.XGBRegressor(**XGB_PARAMS)
    cv.fit(x_df.iloc[:split], y[:split])
    p = cv.predict(x_df.iloc[split:])
    p_ppg = p * (arr[1] - arr[0]) + arr[0]
    a_ppg = y[split:] * (arr[1] - arr[0]) + arr[0]
    mae = mean_absolute_error(a_ppg, p_ppg)
    corr = 0.0
    if len(a_ppg) > 2:
        corr, _ = pearsonr(p_ppg, a_ppg)
        print(f"  Holdout MAE: {mae:.2f} PPG | r={corr:.3f} | n={len(a_ppg)}")
    final = xgb.XGBRegressor(**XGB_PARAMS)
    final.fit(x_df, y)
    return (mae, corr, final)

print("Training RB models (3 PPR formats) with trial-and-error year ranges...")
for ppr in [0, 1, 2]:
    lbl = {0: 'Standard', 1: 'HalfPPR', 2: 'FullPPR'}[ppr]
    print(f"\nRB {lbl}:")

    dfFC_base = dfFantasy.copy()
    dfFC_base = correctData(dfFC_base, ppr)
    dfFC_base = makeCorrectShift(dfFC_base)

    best_corr = -999
    best_model = None
    best_start = None

    for start_year in TRAIN_START_YEARS:
        dfFC = dfFC_base[dfFC_base['season'] >= start_year].copy()
        dfFC = dfFC.loc[dfFC['season'] != 2012]
        dfFC['age_from_peak'] = dfFC['age'] - PEAK_AGE_RB
        dfFC['age_squared'] = dfFC['age'] ** 2
        dfFC['games_missed'] = (17 - dfFC['GP']).clip(lower=0)
        dfFC = removeUnwanted(dfFC, 'RB')
        dfFC = dfFC.reset_index(drop=True)
        if len(dfFC) < 20:
            print(f"  start_year={start_year}: too few rows ({len(dfFC)}), skipping")
            continue
        scaleRB = getScaleBack(dfFC)
        dfFC_scaled = dfFC.copy()
        dfFC_scaled[dfFC_scaled.columns] = scaler.fit_transform(dfFC_scaled[dfFC_scaled.columns])
        print(f"  start_year={start_year} ({len(dfFC)} rows, {len(dfFC.columns)-1} features):", end=" ")
        mae, corr, model = machineLearning(dfFC_scaled, scaleRB)
        if corr > best_corr:
            best_corr = corr
            best_model = model
            best_start = start_year

    print(f"  Best start_year={best_start} with r={best_corr:.3f}")
    path = {0: "rb models/rbModelNonPPR.joblib",
            1: "rb models/rbModelHalfPPR.joblib",
            2: "rb models/rbModelPPR.joblib"}[ppr]
    joblib.dump(best_model, path)
    print(f"  Saved {path}")


Fetching RB opportunity/EPA features for 2015-2025...
  target_share coverage: 100%
  rushing_epa coverage:  91%
Training RB models (3 PPR formats) with trial-and-error year ranges...

RB Standard:
  start_year=2019 (298 rows, 32 features):   Holdout MAE: 2.39 PPG | r=0.347 | n=60
  start_year=2018 (350 rows, 32 features):   Holdout MAE: 2.23 PPG | r=0.410 | n=70
  start_year=2017 (402 rows, 32 features):   Holdout MAE: 2.17 PPG | r=0.526 | n=81
  start_year=2016 (447 rows, 32 features):   Holdout MAE: 2.19 PPG | r=0.540 | n=90
  start_year=2015 (447 rows, 32 features):   Holdout MAE: 2.19 PPG | r=0.540 | n=90
  Best start_year=2016 with r=0.540
  Saved rb models/rbModelNonPPR.joblib

RB HalfPPR:
  start_year=2019 (329 rows, 32 features):   Holdout MAE: 2.75 PPG | r=0.344 | n=66
  start_year=2018 (387 rows, 32 features):   Holdout MAE: 2.45 PPG | r=0.437 | n=78
  start_year=2017 (446 rows, 32 features):   Holdout MAE: 2.44 PPG | r=0.518 | n=90
  start_year=2016 (496 rows, 32 features):

In [3]:
#WR + TE ML MODELS - IMPROVED (separate models per position)
import pandas as pd
import numpy as np
import warnings
import nflreadpy
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
import joblib

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

scaler = MinMaxScaler()
PEAK_AGE_WRTE = 26

dfFantasy = pd.read_pickle("PickleFiles/final_wrte_data.pkl")
dfFantasy.replace([np.inf, -np.inf], np.nan, inplace=True)
for column in dfFantasy.select_dtypes(include=[np.number]).columns:
    dfFantasy[column].fillna(dfFantasy[column].mean(), inplace=True)

# ---- Add opportunity + EPA features from nflreadpy ----
print("Fetching WR/TE opportunity/EPA features for 2015-2025...")
seasonal_all = nflreadpy.load_player_stats(list(range(2015, 2026)), summary_level='reg').to_pandas()
opp = seasonal_all[['player_id', 'season', 'target_share', 'air_yards_share',
                     'wopr', 'racr', 'receiving_epa']].copy()
opp['season'] = opp['season'].astype(int)
dfFantasy['season'] = dfFantasy['season'].astype(int)
dfFantasy = dfFantasy.merge(opp, on=['player_id', 'season'], how='left')
print(f"  target_share coverage: {dfFantasy['target_share'].notna().mean()*100:.0f}%")
print(f"  receiving_epa coverage: {dfFantasy['receiving_epa'].notna().mean()*100:.0f}%")

def correctData(df, pprTF):
    count_cols = ['carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                  'rushing_first_downs', 'rushing_2pt_conversions', 'receptions', 'targets',
                  'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost',
                  'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs',
                  'receiving_2pt_conversions', 'special_teams_tds', 'fantasy_points', 'rrtd', 'age']
    if pprTF == 0:
        df.loc[:, "fantasy_points"] = df["fantasy_points"] - df["receptions"]
    elif pprTF == 1:
        df.loc[:, "fantasy_points"] = df["fantasy_points"] - (df["receptions"] / 2)
    df.loc[:, 'PPG'] = df['fantasy_points'] / df['GP']
    df = df.sort_values(['player_display_name', 'season']).reset_index(drop=True)
    df['ppg_prev'] = df['PPG']
    for col in count_cols:
        df.loc[:, col] = df[col] / df['GP']
    # receiving_epa is season total -> per game; wopr, racr are ratios
    gp_safe = df['GP'].replace(0, np.nan)
    df.loc[:, 'receiving_epa'] = df['receiving_epa'] / gp_safe
    # wopr, racr: do NOT divide
    df = df[df.GP > 7]
    df = df[df.fantasy_points >= 0]
    df = df[df.PPG > 2]
    df = df.sort_values(['player_display_name', 'season']).reset_index(drop=True)
    df['ppg_last_year'] = df.groupby('player_display_name')['PPG'].shift(1).fillna(df['PPG'])
    df['delta_ppg'] = df['PPG'] - df['ppg_last_year']
    return df

def removeUnwanted(dfPos, pos):
    drop_cols = ['season', 'GP', 'season_type', 'fantasy_points',
                 'player_display_name', 'player_id', 'team', 'position']
    return dfPos.drop(columns=drop_cols, errors='ignore')

def makeCorrectShift(df):
    shifters = ['player_id', 'season', 'player_display_name', 'team', 'GP', 'position',
                'age', 'season_type', 'carries', 'rushing_yards', 'rushing_tds',
                'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_2pt_conversions',
                'receptions', 'targets', 'receiving_yards', 'receiving_tds',
                'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch',
                'receiving_first_downs', 'receiving_2pt_conversions', 'special_teams_tds',
                'fantasy_points', 'rrtd',
                'target_share', 'air_yards_share', 'wopr', 'racr',
                'receiving_epa',
                'ppg_prev', 'ppg_last_year', 'delta_ppg']
    shifters = [c for c in shifters if c in df.columns]
    df[shifters] = df.groupby('player_display_name')[shifters].shift(1)
    return df.dropna()

XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 3, 'min_child_weight': 5,
    'reg_alpha': 0.3, 'reg_lambda': 2.0, 'subsample': 0.75,
    'colsample_bytree': 0.7, 'learning_rate': 0.05,
    'random_state': 42, 'verbosity': 0
}

TRAIN_START_YEARS = [2019, 2018, 2017, 2016, 2015]

def getScaleBack(df):
    return [df['PPG'].min(), df['PPG'].max()]

def machineLearning(df, arr):
    predictors = [c for c in df.columns
                  if c != 'PPG' and 'Unnamed' not in c and c != 'YearsBack']
    x_df = df[predictors]
    y = df['PPG'].values
    split = int(len(y) * 0.8)
    cv = xgb.XGBRegressor(**XGB_PARAMS)
    cv.fit(x_df.iloc[:split], y[:split])
    p = cv.predict(x_df.iloc[split:])
    p_ppg = p * (arr[1] - arr[0]) + arr[0]
    a_ppg = y[split:] * (arr[1] - arr[0]) + arr[0]
    mae = mean_absolute_error(a_ppg, p_ppg)
    corr = 0.0
    if len(a_ppg) > 2:
        corr, _ = pearsonr(p_ppg, a_ppg)
        print(f"  Holdout MAE: {mae:.2f} PPG | r={corr:.3f} | n={len(a_ppg)}")
    final = xgb.XGBRegressor(**XGB_PARAMS)
    final.fit(x_df, y)
    return (mae, corr, final)

print("Training WR and TE models separately (3 PPR formats each) with trial-and-error year ranges...")

# Prepare base data (correctData + shift) once, then split by position
for pos_group in ['WR', 'TE']:
    print(f"\n{'='*50}")
    print(f"Position group: {pos_group}")
    for ppr in [0, 1, 2]:
        lbl = {0: 'Standard', 1: 'HalfPPR', 2: 'FullPPR'}[ppr]
        print(f"\n  {pos_group} {lbl}:")

        dfFC_base = dfFantasy[dfFantasy['position'] == pos_group].copy()
        dfFC_base = correctData(dfFC_base, ppr)
        dfFC_base = makeCorrectShift(dfFC_base)

        best_corr = -999
        best_model = None
        best_start = None

        for start_year in TRAIN_START_YEARS:
            dfFC = dfFC_base[dfFC_base['season'] >= start_year].copy()
            dfFC = dfFC.loc[dfFC['season'] != 2012]
            dfFC['age_from_peak'] = dfFC['age'] - PEAK_AGE_WRTE
            dfFC['age_squared'] = dfFC['age'] ** 2
            dfFC['games_missed'] = (17 - dfFC['GP']).clip(lower=0)
            dfFC = removeUnwanted(dfFC, pos_group)
            dfFC = dfFC.reset_index(drop=True)
            if len(dfFC) < 20:
                print(f"    start_year={start_year}: too few rows ({len(dfFC)}), skipping")
                continue
            scale_arr = getScaleBack(dfFC)
            dfFC_scaled = dfFC.copy()
            dfFC_scaled[dfFC_scaled.columns] = scaler.fit_transform(dfFC_scaled[dfFC_scaled.columns])
            print(f"    start_year={start_year} ({len(dfFC)} rows, {len(dfFC.columns)-1} features):", end=" ")
            mae, corr, model = machineLearning(dfFC_scaled, scale_arr)
            if corr > best_corr:
                best_corr = corr
                best_model = model
                best_start = start_year

        print(f"    Best start_year={best_start} with r={best_corr:.3f}")
        prefix = 'wr' if pos_group == 'WR' else 'te'
        path = {0: f"wrte models/{prefix}ModelNonPPR.joblib",
                1: f"wrte models/{prefix}ModelHalfPPR.joblib",
                2: f"wrte models/{prefix}ModelPPR.joblib"}[ppr]
        joblib.dump(best_model, path)
        print(f"    Saved {path}")


Fetching WR/TE opportunity/EPA features for 2015-2025...
  target_share coverage: 100%
  receiving_epa coverage: 94%
Training WR and TE models separately (3 PPR formats each) with trial-and-error year ranges...

Position group: WR

  WR Standard:
    start_year=2019 (294 rows, 34 features):   Holdout MAE: 1.10 PPG | r=0.490 | n=59
    start_year=2018 (351 rows, 34 features):   Holdout MAE: 1.07 PPG | r=0.500 | n=71
    start_year=2017 (395 rows, 34 features):   Holdout MAE: 1.11 PPG | r=0.489 | n=79
    start_year=2016 (453 rows, 34 features):   Holdout MAE: 1.06 PPG | r=0.475 | n=91
    start_year=2015 (453 rows, 34 features):   Holdout MAE: 1.06 PPG | r=0.475 | n=91
    Best start_year=2018 with r=0.500
    Saved wrte models/wrModelNonPPR.joblib

  WR HalfPPR:
    start_year=2019 (468 rows, 34 features):   Holdout MAE: 1.31 PPG | r=0.717 | n=94
    start_year=2018 (551 rows, 34 features):   Holdout MAE: 1.26 PPG | r=0.705 | n=111
    start_year=2017 (622 rows, 34 features):   Holdout